# Thử nghiệm khóa chữ số

Bước 9 của kế hoạch S6E11. [Kế hoạch dự án](../../../plan.md#L298)

D01 thêm 6 giá trị phần trăm của đơn vị, chẳng hạn 37 trong 1371.37. D02 mã hóa theo nhãn 36 khóa chữ số từ sáu cột số. D03 kết hợp hai nhóm. Mỗi lượt giữ cùng mô hình TE làm đối chứng. LightGBM dùng T04 và XGBoost dùng T03.

## Chạy lại

Chạy từ thư mục `s6e11_gpu/plan_runs` với môi trường Python của dự án và GPU đã kiểm tra. Đặt `S6E11_DATA_DIR` hoặc thêm `--data-dir` nếu CSV chính thức ở thư mục khác. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```sh
python -B -X utf8 run_plan.py --stage features --compare-xgb-digits
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "INT3405E_FD" / "s6e11_gpu" / "plan_runs",
                  base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "INT3405E_FD" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()
                 and (path / "report_results.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'features', '--compare-xgb-digits']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

Nguồn số liệu là bản tổng hợp công bố từ lượt chạy GPU lịch sử trong `published/full`. Chế độ đọc bản công bố đối chiếu các bảng tổng hợp và mã kiểm tra đã ghi. Tính lại chỉ số từ từng dự đoán cần dữ liệu và kết quả đầy đủ ở máy chạy. [Bảng tổng hợp](../published/full/comparison.csv)

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `XGB-D03` | `T03` | 0.950182 | +0.001688 | 0.528763 | 0.044422 | 231.4 | 0.949852 | 0.950884 | 0.954957 | 0.950990 | 0.944603 |
| `D03` | `T04` | 0.950131 | +0.001557 | 0.527604 | 0.044475 | 871.3 | 0.949598 | 0.951024 | 0.954852 | 0.951197 | 0.944163 |
| `D02` | `T04` | 0.950124 | +0.001550 | 0.528206 | 0.044435 | 1115.7 | 0.949649 | 0.951012 | 0.954810 | 0.951146 | 0.944175 |
| `XGB-D02` | `T03` | 0.950074 | +0.001580 | 0.528382 | 0.044444 | 217.4 | 0.949728 | 0.950997 | 0.954809 | 0.950887 | 0.944387 |
| `D01` | `T04` | 0.948784 | +0.000210 | 0.521507 | 0.044890 | 898.4 | 0.947576 | 0.949905 | 0.954315 | 0.949367 | 0.943274 |
| `XGB-D01` | `T03` | 0.948623 | +0.000129 | 0.521631 | 0.044920 | 157.2 | 0.947424 | 0.949972 | 0.953856 | 0.949065 | 0.943195 |

## Đọc dữ liệu kết quả hiện tại

Cell này ưu tiên dự đoán OOF và tệp metric cục bộ khi có lượt chạy. Bản sao mới đọc bảng tổng hợp trong `published/full`.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import ARCHIVE_PROVENANCE, final_summary, read_stage, result_source

result_dir, archived = result_source()
if archived:
    display(Markdown(ARCHIVE_PROVENANCE))
else:
    display(Markdown("Nguồn cục bộ. Chỉ số của lượt hoàn tất được tính lại từ dự đoán OOF."))
rows, pending, notes = read_stage('02-digit', result_dir, archived=archived)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if False:
    display(Markdown(final_summary(result_dir, markdown_links=False, archived=archived)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    if not archived:
        with np.load(result_dir / record["name"] / "predictions.npz", allow_pickle=False) as saved:
            display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

Trình chạy đo D03 cùng D01 và D02 để so sánh trên cùng fold. Kết quả chữ số gắn với cấu trúc của bộ dữ liệu tổng hợp.